In [1]:
import numpy as np
import matplotlib.pyplot as plt


def read_latency_data(file_path, ignore):
    with open(file_path, "r") as file:
        latencies = [int(line.strip()) for line in file if line.strip().isdigit()]
    n_ignore = int(len(latencies) * ignore)
    latencies = latencies[n_ignore:]
    return latencies


def get_pctl(data, pctl):
    data = np.sort(data)
    tail = np.percentile(data, pctl).astype(int)
    return tail

In [2]:
# rps_values = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000]
rps_values = [1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000]

datas: dict = {}

for pctl in [50, 90, 95, 99, 99.9]:
    data = {
        "rps": [],
        "fcfs": [],
        "masa": [],
        "relative": [],
    }
    for rps in rps_values:
        data["rps"].append(rps)

        file_path = f"r{rps}-fcfs.csv"
        tail = get_pctl(read_latency_data(file_path, ignore=0.2), pctl)
        data["fcfs"].append(tail)

        file_path = f"r{rps}-masa.csv"
        tail = get_pctl(read_latency_data(file_path, ignore=0.2), pctl)
        data["masa"].append(tail)

        data["relative"].append((data["fcfs"][-1] - data["masa"][-1]) / data["fcfs"][-1] * 100)
    datas[pctl] = data

print(datas)

{50: {'rps': [1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000], 'fcfs': [37912, 42888, 48452, 56446, 67231, 83624, 115555, 173299, 322903, 754748], 'masa': [39307, 44608, 50429, 58995, 70424, 87099, 120020, 178136, 335440, 778350], 'relative': [-3.6795737497362313, -4.0104458123484426, -4.080326921489309, -4.5158204301456255, -4.749297199208699, -4.155505596479479, -3.8639608844273288, -2.7911297814759464, -3.8825901276854045, -3.12713647469089]}, 90: {'rps': [1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000], 'fcfs': [62233, 70322, 79345, 92217, 110769, 136271, 189883, 282229, 525437, 1181374], 'masa': [59084, 65612, 73096, 83943, 100014, 121198, 168802, 244332, 461596, 985680], 'relative': [5.060016390018158, 6.697761724638093, 7.875732560337766, 8.972315299782036, 9.709395227906725, 11.061047471582363, 11.102099714034432, 13.427748388719799, 12.150076983539416, 16.56494894927432]}, 95: {'rps': [1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000], 'fcfs':

In [3]:
import pandas as pd

for pctl in [90, 95, 99, 99.9]:
    data = datas[pctl]
    data["decrease"] = data["relative"]
    data["improve"] = [1.0 / (1.0 - x / 100) for x in data["relative"]]

    data["decrease"] = [f"{round(x, 2)}%" for x in data["decrease"]]
    data["improve"] = [f"{round(x, 2)}%" for x in data["improve"]]

    df = pd.DataFrame(data)
    df = df[["rps", "decrease", "improve"]]
    print(f"tail: {pctl}%")
    print(df)
    print()

tail: 90%
    rps decrease improve
0  1100    5.06%   1.05%
1  1200     6.7%   1.07%
2  1300    7.88%   1.09%
3  1400    8.97%    1.1%
4  1500    9.71%   1.11%
5  1600   11.06%   1.12%
6  1700    11.1%   1.12%
7  1800   13.43%   1.16%
8  1900   12.15%   1.14%
9  2000   16.56%    1.2%

tail: 95%
    rps decrease improve
0  1100    7.82%   1.08%
1  1200    9.58%   1.11%
2  1300   11.17%   1.13%
3  1400   12.02%   1.14%
4  1500   12.64%   1.14%
5  1600   14.27%   1.17%
6  1700    14.5%   1.17%
7  1800   16.14%   1.19%
8  1900   15.48%   1.18%
9  2000   18.41%   1.23%

tail: 99%
    rps decrease improve
0  1100   12.01%   1.14%
1  1200   13.98%   1.16%
2  1300   15.72%   1.19%
3  1400    16.2%   1.19%
4  1500   15.02%   1.18%
5  1600   18.12%   1.22%
6  1700   21.19%   1.27%
7  1800   20.35%   1.26%
8  1900   20.22%   1.25%
9  2000   21.97%   1.28%

tail: 99.9%
    rps decrease improve
0  1100   15.19%   1.18%
1  1200   15.75%   1.19%
2  1300   20.11%   1.25%
3  1400   20.84%   1.26%
4  15

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["font.family"] = "Roboto"

plt.rcParams.update(
    {
        "font.size": 20,  # Sets the base default font size
        "axes.labelsize": 20,  # Font size for x and y labels
        "axes.titlesize": 20,  # Font size for plot title
        "xtick.labelsize": 20,  # Font size for x-axis tick labels
        "ytick.labelsize": 20,  # Font size for y-axis tick labels
        "legend.fontsize": 20,  # Font size for legend
    }
)

data = raw

for key in ["fcfs", "masa"]:
    data[key] = [x / 1e3 for x in data[key]]

data["diff"] = []
for i in range(len(data["rps"])):
    data["diff"].append(data["masa"][i] - data["fcfs"][i])

print(data)

# print(data)

# Setting the positions of the bars on the x-axis
x = np.arange(len(data["rps"]))  # the label locations
width = 0.3  # the width of the bars

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(
    x - width / 2,
    data["fcfs"],
    width,
    label="FCFS",
    color="#C25759",
)
rects2 = ax.bar(
    x + width / 2,
    data["masa"],
    width,
    label="Masa",
    color="#599CB4",
)

# Annotate the bars with the diff
# for i, diff in enumerate(data["diff"]):
#     if diff > 0:
#         text = f"+{diff:.1f}"
#     else:
#         text = f"{diff:.1f}"
#     ax.text(
#         x[i] + width / 2,
#         max(data["fcfs"][i], data["masa"][i]) + 0.2,
#         text,
#         ha="center",
#         va="bottom",
#         color="black",
#         fontsize=18,
#     )

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_title("Elapse = 0")
ax.set_xlabel("RPS")
ax.set_ylabel("P99 Latency (ms)")
# ax.set_ylim(0, 40)
ax.set_xticks(x)
ax.set_xticklabels(data["rps"])
ax.legend()

fig.tight_layout()
plt.savefig("ex0.pdf")
plt.show()